In [1]:
import sys
import os
sys.path
sys.path.append('..')
import pandas as pd
import numpy as np
import torch
import pytorch_lightning as pl
from pytorch_lightning.loggers import TensorBoardLogger
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

In [2]:
# Import your model and dataset class
from utilities import EfficientNetMultiOutput
from utilities import PreprocessedDataset
from utilities import UnfreezeCallbackEffNet

In [3]:
torch.set_float32_matmul_precision('medium')

In [4]:
preprocessed_dir = '../data/preprocessed_EDA'
cleaned_csv = "../data/cleaned_data.csv"
df = pd.read_csv(cleaned_csv)

In [5]:
df.sample(5)

,filename,age,gender
18425,55_0_3_20170119171728191.jpg.chip.jpg,55,0
21661,80_1_0_20170110153307958.jpg.chip.jpg,80,1
19666,61_0_0_20170104213537981.jpg.chip.jpg,61,0
6300,26_0_4_20170116193007097.jpg.chip.jpg,26,0
13617,37_0_3_20170119202312469.jpg.chip.jpg,37,0


In [6]:
train_df, temp_df = train_test_split(df, test_size=0.3, random_state=42, stratify=df['gender'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['gender'])


In [7]:
print(f"Training set size: {len(train_df)}")
print(f"Validation set size: {len(val_df)}")
print(f"Test set size: {len(test_df)}")

Training set size: 15807
Validation set size: 3387
Test set size: 3388


In [8]:
num_workers = 2

train_dataset = PreprocessedDataset(train_df, preprocessed_dir)
val_dataset = PreprocessedDataset(val_df, preprocessed_dir)
test_dataset = PreprocessedDataset(test_df, preprocessed_dir)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=num_workers, pin_memory=True, persistent_workers=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=num_workers, pin_memory=True, persistent_workers=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=num_workers, pin_memory=True, persistent_workers=True)


In [9]:
gender_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1]),
    y=train_df['gender']
)
gender_weights = torch.tensor(gender_weights, dtype=torch.float32)

class_weights = {'gender': gender_weights}

In [10]:
logger = TensorBoardLogger('../tb_logs', name='EffNet_regression')

# Checkpoint callback to save the best model
checkpoint_callback = ModelCheckpoint(
    monitor='val_loss',
    filename='EffNet-best-{epoch:02d}-{val_loss:.2f}',
    save_top_k=1,
    mode='min',
    dirpath='../models/'
)

# Early stopping to prevent overfitting
early_stop_callback = EarlyStopping(
    monitor='val_loss',
    patience=10,
    mode='min'
)

# Import and create your custom unfreeze callback
unfreeze_callback = UnfreezeCallbackEffNet(
    patience=7,    # how many epochs of no val_loss improvement before unfreezing
    min_delta=0.0  # minimum improvement considered as progress
)

# Setup Trainer with all callbacks included
trainer = pl.Trainer(
    max_epochs=99,
    accelerator='gpu',
    devices=1,
    logger=logger,
    callbacks=[checkpoint_callback, early_stop_callback, unfreeze_callback],
    precision='16-mixed',  # recommended format instead of deprecated '16'
    accumulate_grad_batches=4,
    log_every_n_steps=10
)

Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [11]:
model = EfficientNetMultiOutput(class_weights=class_weights, lr=1e-3)

In [12]:
trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=val_loader)

C:\Users\MiestoMeska\anaconda3\envs\TC_4\lib\site-packages\pytorch_lightning\callbacks\model_checkpoint.py:654: Checkpoint directory C:\Users\MiestoMeska\Desktop\Mokslai\VIKO\BD\CNN_AgeGenderRace\models exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
C:\Users\MiestoMeska\anaconda3\envs\TC_4\lib\site-packages\torch\optim\lr_scheduler.py:28: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn("The verbose parameter is deprecated. Please use get_last_lr() "

  | Name              | Type                | Params | Mode 
------------------------------------------------------------------
0 | efficientnet      | EfficientNet        | 4.0 M  | train
1 | gender_classifier | Sequential          | 2.0 M  | train
2 | age_regressor     | Sequential          | 2.0 M  | train
3 | criterion_gender  | CrossEntropyLoss    | 0      | train
4 | criterion_age     | MSELoss             | 0      | train
5 | gender_accuracy 

Sanity Checking: |                                                                               | 0/? [00:00<…

Training: |                                                                                      | 0/? [00:00<…

Validation: |                                                                                    | 0/? [00:00<…

Validation: |                                                                                    | 0/? [00:00<…

...

Validation: |                                                                                    | 0/? [00:00<…

🔓 Unfreezing all layers due to plateau in validation loss.
⚙️ Learning rate adjusted from 0.000300 → 0.000090
✅ Layers unfrozen: all, optimizer reinitialized, learning rate updated.


Validation: |                                                                                    | 0/? [00:00<…

Validation: |                                                                                    | 0/? [00:00<…

Validation: |                                                                                    | 0/? [00:00<…